# Stock Market Trend Visualizer: Interactive Multi-Stock Analysis

## Project Overview

This analysis explores stock market trends for major tech companies using interactive visualizations built with Plotly. The project demonstrates:

- **Real-time data acquisition** via Yahoo Finance API with local caching
- **Time-series manipulation** including technical indicator computation
- **Interactive visualizations** with hover tooltips, zoom, and range selection
- **Financial analysis** covering risk metrics, correlation, and trend identification

**Tickers:** AAPL, TSLA, MSFT, GOOGL, AMZN  
**Period:** 2 Years of Daily Data  
**Author:** [Your Name]  
**Date:** February 2026  
**Portfolio Project:** Day 2 of 30

---

## Table of Contents

1. [Configuration and Setup](#1-configuration-and-setup)
2. [Data Acquisition](#2-data-acquisition)
3. [Data Preprocessing and Feature Engineering](#3-data-preprocessing-and-feature-engineering)
4. [Candlestick Chart with Volume](#4-candlestick-chart-with-volume)
5. [Moving Averages Analysis](#5-moving-averages-analysis)
6. [Multi-Stock Price Comparison](#6-multi-stock-price-comparison)
7. [Returns Analysis](#7-returns-analysis)
8. [Volatility and Correlation](#8-volatility-and-correlation)
9. [Volume Analysis](#9-volume-analysis)
10. [Statistics Dashboard](#10-statistics-dashboard)
11. [Key Findings](#11-key-findings)

## 1. Configuration and Setup

In [ ]:
# ============================================================
# CONFIGURATION - Modify these settings to customize analysis
# ============================================================
TICKERS = ['AAPL', 'TSLA', 'MSFT', 'GOOGL', 'AMZN']
PERIOD = '2y'                    # Data period: 1y, 2y, 5y, 10y, max
INTERVAL = '1d'                  # Data interval: 1d, 1wk, 1mo
CACHE_DATA = True                # Save data locally for offline use
# ============================================================

import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from datetime import datetime, timedelta
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Color palette - brand colors for each ticker
COLORS = {
    'AAPL': '#007AFF',
    'TSLA': '#CC0000',
    'MSFT': '#00A4EF',
    'GOOGL': '#34A853',
    'AMZN': '#FF9900',
}
DEFAULT_COLORS = px.colors.qualitative.Set2

def get_color(ticker, idx=0):
    """Get color for a ticker, falling back to default palette."""
    return COLORS.get(ticker, DEFAULT_COLORS[idx % len(DEFAULT_COLORS)])

# Plotly styling constants
PLOTLY_TEMPLATE = 'plotly_white'
CHART_HEIGHT = 600
CHART_WIDTH = 1000

# Ensure output directories exist
os.makedirs('../data', exist_ok=True)
os.makedirs('../visualizations', exist_ok=True)

print('Libraries loaded successfully!')
print(f'Analyzing tickers: {", ".join(TICKERS)}')
print(f'Period: {PERIOD}, Interval: {INTERVAL}')

## 2. Data Acquisition

We use the `yfinance` library to fetch historical stock data from Yahoo Finance. The download function includes:
- **Per-ticker error handling** so one failure doesn't block others
- **Local CSV caching** to avoid repeated API calls during development
- **Automatic MultiIndex handling** for yfinance column format changes

In [ ]:
def download_stock_data(tickers, period='2y', interval='1d', cache=True):
    """
    Download stock data from Yahoo Finance with error handling and caching.
    
    Parameters
    ----------
    tickers : list
        List of stock ticker symbols.
    period : str
        Data period (e.g., '1y', '2y', '5y').
    interval : str
        Data interval (e.g., '1d', '1wk').
    cache : bool
        Whether to save/load from local CSV cache.
        
    Returns
    -------
    dict
        Dictionary mapping ticker -> DataFrame.
    """
    stock_data = {}
    failed_tickers = []
    
    for ticker in tickers:
        cache_path = f'../data/{ticker}_{period}_{interval}.csv'
        
        # Try loading from cache first
        if cache:
            try:
                df = pd.read_csv(cache_path, index_col='Date', parse_dates=True)
                print(f'  {ticker}: Loaded from cache ({len(df):,} rows)')
                stock_data[ticker] = df
                continue
            except FileNotFoundError:
                pass
        
        # Download from Yahoo Finance
        try:
            print(f'  {ticker}: Downloading from Yahoo Finance...', end=' ')
            df = yf.download(ticker, period=period, interval=interval,
                           progress=False, auto_adjust=False)
            
            if df.empty:
                raise ValueError(f'No data returned for {ticker}')
            
            # Flatten MultiIndex columns if present
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            
            stock_data[ticker] = df
            print(f'OK ({len(df):,} rows, {df.index[0].strftime("%Y-%m-%d")} to {df.index[-1].strftime("%Y-%m-%d")})')
            
            # Cache locally
            if cache:
                df.to_csv(cache_path)
                
        except Exception as e:
            failed_tickers.append(ticker)
            print(f'FAILED ({str(e)})')
    
    if failed_tickers:
        print(f'\nWarning: Failed to download: {", ".join(failed_tickers)}')
    
    print(f'\nSuccessfully loaded {len(stock_data)}/{len(tickers)} tickers')
    return stock_data


# Download all stock data
print('Downloading stock data...')
print('=' * 60)
stock_data = download_stock_data(TICKERS, period=PERIOD, interval=INTERVAL, cache=CACHE_DATA)

In [ ]:
# Inspect downloaded data
print('DATA SUMMARY')
print('=' * 70)

for ticker, df in stock_data.items():
    print(f'\n{ticker}:')
    print(f'  Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
    print(f'  Date Range: {df.index[0].strftime("%Y-%m-%d")} to {df.index[-1].strftime("%Y-%m-%d")}')
    print(f'  Columns: {list(df.columns)}')
    print(f'  Latest Close: ${df["Close"].iloc[-1]:.2f}')

# Show sample data for first ticker
first_ticker = list(stock_data.keys())[0]
print(f'\nSample data ({first_ticker}):')
stock_data[first_ticker].tail()

In [ ]:
# Data quality assessment
print('DATA QUALITY REPORT')
print('=' * 70)

for ticker, df in stock_data.items():
    missing = df.isnull().sum()
    missing_pct = df.isnull().sum() / len(df) * 100
    
    print(f'\n{ticker}:')
    print(f'  Total trading days: {len(df):,}')
    print(f'  Missing values:')
    has_missing = False
    for col in df.columns:
        if missing[col] > 0:
            print(f'    {col}: {missing[col]} ({missing_pct[col]:.2f}%)')
            has_missing = True
    if not has_missing:
        print(f'    None detected')
    
    # Check for zero volumes
    zero_vol = (df['Volume'] == 0).sum()
    if zero_vol > 0:
        print(f'  Zero-volume days: {zero_vol}')

## 3. Data Preprocessing and Feature Engineering

We compute a set of derived features commonly used in quantitative finance:

| Feature | Description | Use Case |
|---------|-------------|----------|
| Daily Return | % change in adj. close | Performance measurement |
| Log Return | ln(price_t / price_t-1) | Statistical analysis (additive property) |
| SMA 20/50/200 | Simple Moving Averages | Trend identification |
| EMA 20 | Exponential Moving Average | Momentum tracking |
| Rolling Volatility | 21-day annualized std dev | Risk measurement |
| Normalized Price | Base-100 indexed price | Cross-stock comparison |

In [ ]:
def preprocess_stock_data(stock_data):
    """
    Preprocess stock data: handle missing values and compute derived features.
    
    Parameters
    ----------
    stock_data : dict
        Dictionary of ticker -> DataFrame.
        
    Returns
    -------
    dict
        Processed stock data with additional columns.
    """
    processed = {}
    
    for ticker, df in stock_data.items():
        df = df.copy()
        
        # Handle missing values
        df.ffill(inplace=True)
        df.bfill(inplace=True)
        
        # Daily returns
        df['Daily_Return'] = df['Adj Close'].pct_change()
        
        # Log returns (additive across time periods)
        df['Log_Return'] = np.log(df['Adj Close'] / df['Adj Close'].shift(1))
        
        # Cumulative return from start of period
        df['Cumulative_Return'] = (1 + df['Daily_Return']).cumprod() - 1
        
        # Simple Moving Averages
        df['SMA_20'] = df['Adj Close'].rolling(window=20).mean()
        df['SMA_50'] = df['Adj Close'].rolling(window=50).mean()
        df['SMA_200'] = df['Adj Close'].rolling(window=200).mean()
        
        # Exponential Moving Average
        df['EMA_20'] = df['Adj Close'].ewm(span=20, adjust=False).mean()
        
        # Rolling volatility (21 trading days ~ 1 month, annualized with sqrt(252))
        df['Volatility_21d'] = df['Daily_Return'].rolling(window=21).std() * np.sqrt(252)
        
        # Normalized price for cross-stock comparison (base = 100)
        df['Normalized_Price'] = df['Adj Close'] / df['Adj Close'].iloc[0] * 100
        
        # Volume moving average
        df['Volume_SMA_20'] = df['Volume'].rolling(window=20).mean()
        
        processed[ticker] = df
    
    return processed


# Process all stock data
stock_data = preprocess_stock_data(stock_data)

print('Feature engineering complete!')
print('New columns: Daily_Return, Log_Return, Cumulative_Return,')
print('  SMA_20, SMA_50, SMA_200, EMA_20, Volatility_21d,')
print('  Normalized_Price, Volume_SMA_20')

In [ ]:
# Build combined DataFrames for cross-stock analysis (inner join on common dates)

active_tickers = list(stock_data.keys())

# Combined adjusted close prices
prices_df = pd.DataFrame({
    ticker: stock_data[ticker]['Adj Close']
    for ticker in active_tickers
})
prices_df.dropna(inplace=True)

# Combined daily returns
returns_df = pd.DataFrame({
    ticker: stock_data[ticker]['Daily_Return']
    for ticker in active_tickers
})
returns_df.dropna(inplace=True)

# Combined normalized prices
normalized_df = pd.DataFrame({
    ticker: stock_data[ticker]['Normalized_Price']
    for ticker in active_tickers
})
normalized_df.dropna(inplace=True)

# Combined volatility
volatility_df = pd.DataFrame({
    ticker: stock_data[ticker]['Volatility_21d']
    for ticker in active_tickers
})
volatility_df.dropna(inplace=True)

print(f'Combined price data: {prices_df.shape[0]:,} trading days x {prices_df.shape[1]} stocks')
print(f'Date range: {prices_df.index[0].strftime("%Y-%m-%d")} to {prices_df.index[-1].strftime("%Y-%m-%d")}')
print(f'\nLatest prices:')
print(prices_df.tail(1).T.to_string())

In [ ]:
# Verify computed features for first ticker
sample_ticker = active_tickers[0]
sample_df = stock_data[sample_ticker]

print(f'FEATURE VERIFICATION: {sample_ticker}')
print('=' * 60)
print(f'Total rows: {len(sample_df):,}')
print(f'\nNaN counts after preprocessing:')
print(sample_df[['Daily_Return', 'SMA_20', 'SMA_50', 'SMA_200', 'Volatility_21d']].isnull().sum())
print(f'\nSample of computed features (last 5 days):')
sample_df[['Adj Close', 'Daily_Return', 'SMA_20', 'SMA_50', 'Volatility_21d', 'Normalized_Price']].tail()

## 4. Candlestick Chart with Volume

The candlestick chart is the foundational visualization in technical analysis. Each candle shows the open, high, low, and close prices for a single trading day:

- **Green candles**: Close > Open (bullish day, price went up)
- **Red candles**: Close < Open (bearish day, price went down)
- **Wicks/shadows**: Show the high and low extremes of the day

Volume bars below show trading activity intensity. High volume during price increases suggests strong buying conviction.

In [ ]:
def create_candlestick_volume(df, ticker, last_n_days=120):
    """
    Create an interactive candlestick chart with volume overlay.
    
    Parameters
    ----------
    df : pd.DataFrame
        Stock data with OHLCV columns.
    ticker : str
        Stock ticker symbol.
    last_n_days : int
        Number of recent trading days to display (default 120 ~ 6 months).
    """
    plot_df = df.tail(last_n_days)
    
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=(f'{ticker} Price', 'Volume'),
        row_heights=[0.7, 0.3]
    )
    
    # Candlestick
    fig.add_trace(
        go.Candlestick(
            x=plot_df.index,
            open=plot_df['Open'],
            high=plot_df['High'],
            low=plot_df['Low'],
            close=plot_df['Close'],
            name='OHLC',
            increasing_line_color='#26a69a',
            decreasing_line_color='#ef5350'
        ),
        row=1, col=1
    )
    
    # Volume bars colored by price direction
    colors = ['#26a69a' if close >= open_ else '#ef5350'
              for close, open_ in zip(plot_df['Close'], plot_df['Open'])]
    
    fig.add_trace(
        go.Bar(
            x=plot_df.index,
            y=plot_df['Volume'],
            marker_color=colors,
            name='Volume',
            opacity=0.7
        ),
        row=2, col=1
    )
    
    fig.update_layout(
        title=f'{ticker} Stock Price and Volume (Last {last_n_days} Trading Days)',
        template=PLOTLY_TEMPLATE,
        height=CHART_HEIGHT,
        width=CHART_WIDTH,
        xaxis_rangeslider_visible=False,
        showlegend=False,
        yaxis_title='Price ($)',
        yaxis2_title='Volume',
    )
    
    # Range selector buttons
    fig.update_xaxes(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label='1M', step='month', stepmode='backward'),
                dict(count=3, label='3M', step='month', stepmode='backward'),
                dict(count=6, label='6M', step='month', stepmode='backward'),
                dict(step='all', label='All')
            ])
        ),
        row=1, col=1
    )
    
    fig.show()
    fig.write_html(f'../visualizations/01_candlestick_{ticker}.html')
    print(f'Saved: visualizations/01_candlestick_{ticker}.html')


# Generate candlestick for the first ticker
create_candlestick_volume(stock_data[active_tickers[0]], active_tickers[0])

In [ ]:
# Generate candlestick charts for all tickers
# Uncomment the loop below to generate for every ticker:
# for ticker in active_tickers:
#     create_candlestick_volume(stock_data[ticker], ticker)

print('Tip: Call create_candlestick_volume(stock_data["TSLA"], "TSLA") for any ticker')

## 5. Moving Averages Analysis

Moving averages smooth out price fluctuations to reveal underlying trends:

- **SMA 20** (short-term): Captures recent momentum, reacts quickly to price changes
- **SMA 50** (medium-term): Identifies intermediate trends
- **SMA 200** (long-term): Defines the overall market direction; widely watched by institutional investors

### Key Trading Signals
- **Golden Cross**: Short-term MA crosses *above* long-term MA (bullish signal)
- **Death Cross**: Short-term MA crosses *below* long-term MA (bearish signal)

These crossover signals are among the most widely followed indicators in technical analysis.

In [ ]:
def create_moving_averages_chart(df, ticker):
    """
    Create price chart with SMA 20/50/200 overlays and cross signal annotations.
    
    Parameters
    ----------
    df : pd.DataFrame
        Preprocessed stock data.
    ticker : str
        Stock ticker symbol.
    """
    fig = go.Figure()
    
    # Price line
    fig.add_trace(go.Scatter(
        x=df.index, y=df['Adj Close'],
        name='Price',
        line=dict(color=get_color(ticker), width=1.5),
        opacity=0.8
    ))
    
    # SMA lines
    sma_config = [
        ('SMA_20', '20-Day SMA', '#FF6B6B', 1.2, 'dot'),
        ('SMA_50', '50-Day SMA', '#4ECDC4', 1.5, 'dash'),
        ('SMA_200', '200-Day SMA', '#45B7D1', 2.0, 'solid'),
    ]
    
    for col, name, color, width, dash in sma_config:
        fig.add_trace(go.Scatter(
            x=df.index, y=df[col],
            name=name,
            line=dict(color=color, width=width, dash=dash)
        ))
    
    # Detect Golden Cross and Death Cross (SMA 50 vs SMA 200)
    df_signals = df.dropna(subset=['SMA_50', 'SMA_200']).copy()
    df_signals['Signal'] = np.where(df_signals['SMA_50'] > df_signals['SMA_200'], 1, 0)
    df_signals['Cross'] = df_signals['Signal'].diff()
    
    golden_crosses = df_signals[df_signals['Cross'] == 1]
    death_crosses = df_signals[df_signals['Cross'] == -1]
    
    # Annotate golden crosses
    for idx, row in golden_crosses.iterrows():
        fig.add_annotation(
            x=idx, y=row['Adj Close'],
            text='Golden Cross',
            showarrow=True, arrowhead=2, arrowcolor='green',
            font=dict(color='green', size=10),
            bgcolor='rgba(255,255,255,0.8)'
        )
    
    # Annotate death crosses
    for idx, row in death_crosses.iterrows():
        fig.add_annotation(
            x=idx, y=row['Adj Close'],
            text='Death Cross',
            showarrow=True, arrowhead=2, arrowcolor='red',
            font=dict(color='red', size=10),
            bgcolor='rgba(255,255,255,0.8)'
        )
    
    fig.update_layout(
        title=f'{ticker} Price with Moving Averages (20/50/200 Day)',
        xaxis_title='Date',
        yaxis_title='Price ($)',
        template=PLOTLY_TEMPLATE,
        height=CHART_HEIGHT,
        width=CHART_WIDTH,
        hovermode='x unified',
        legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
    )
    
    fig.show()
    fig.write_html(f'../visualizations/02_moving_averages_{ticker}.html')
    print(f'Saved: visualizations/02_moving_averages_{ticker}.html')
    print(f'Golden Crosses detected: {len(golden_crosses)}')
    print(f'Death Crosses detected: {len(death_crosses)}')


create_moving_averages_chart(stock_data[active_tickers[0]], active_tickers[0])

## 6. Multi-Stock Price Comparison

Comparing stocks with vastly different price levels (e.g., AMZN at ~$180 vs TSLA at ~$300) on an absolute scale is misleading. Instead, we **normalize all prices to a base of 100** at the start of the period.

This means:
- A stock at **150** has gained **50%** since the start
- A stock at **80** has lost **20%** since the start
- Regardless of their absolute dollar prices

In [ ]:
def create_normalized_comparison(normalized_df):
    """
    Create normalized price comparison chart for all tickers.
    
    Parameters
    ----------
    normalized_df : pd.DataFrame
        DataFrame with normalized prices (base=100) for each ticker.
    """
    fig = go.Figure()
    
    for i, ticker in enumerate(normalized_df.columns):
        color = get_color(ticker, i)
        total_return = (normalized_df[ticker].iloc[-1] / 100 - 1) * 100
        
        fig.add_trace(go.Scatter(
            x=normalized_df.index,
            y=normalized_df[ticker],
            name=f'{ticker} ({total_return:+.1f}%)',
            line=dict(color=color, width=2),
            hovertemplate=(
                f'{ticker}<br>'
                'Date: %{x}<br>'
                'Indexed: %{y:.1f}<br>'
                'Return: %{customdata:.1f}%'
                '<extra></extra>'
            ),
            customdata=normalized_df[ticker] - 100
        ))
    
    # Baseline at 100
    fig.add_hline(
        y=100, line_dash='dash', line_color='gray', opacity=0.5,
        annotation_text='Baseline (Start)', annotation_position='bottom right'
    )
    
    fig.update_layout(
        title='Normalized Stock Price Comparison (Base = 100)',
        xaxis_title='Date',
        yaxis_title='Normalized Price (Base = 100)',
        template=PLOTLY_TEMPLATE,
        height=CHART_HEIGHT,
        width=CHART_WIDTH,
        hovermode='x unified',
        legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
    )
    
    fig.show()
    fig.write_html('../visualizations/03_normalized_comparison.html')
    print('Saved: visualizations/03_normalized_comparison.html')


create_normalized_comparison(normalized_df)

## 7. Returns Analysis

### Daily Returns Distribution

The distribution of daily returns reveals the **risk profile** of each stock:

- **Wider distribution** = higher volatility = more risk
- **Fat tails** (kurtosis > 3) = more extreme events than a normal distribution predicts
- **Negative skew** = larger losses more likely than gains of equal magnitude

These properties are critical for risk management and portfolio construction.

In [ ]:
def create_returns_distribution(returns_df):
    """
    Create overlaid histogram of daily returns with normal distribution overlay.
    
    Parameters
    ----------
    returns_df : pd.DataFrame
        DataFrame with daily returns for each ticker.
    """
    fig = go.Figure()
    
    for i, ticker in enumerate(returns_df.columns):
        color = get_color(ticker, i)
        returns = returns_df[ticker].dropna()
        
        fig.add_trace(go.Histogram(
            x=returns,
            name=ticker,
            opacity=0.6,
            nbinsx=80,
            marker_color=color,
            histnorm='probability density'
        ))
    
    # Normal distribution overlay for reference (first ticker)
    first_ticker = returns_df.columns[0]
    mu = returns_df[first_ticker].mean()
    sigma = returns_df[first_ticker].std()
    x_range = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 200)
    normal_curve = stats.norm.pdf(x_range, mu, sigma)
    
    fig.add_trace(go.Scatter(
        x=x_range, y=normal_curve,
        name=f'Normal Dist. ({first_ticker})',
        line=dict(color='black', width=2, dash='dash')
    ))
    
    fig.update_layout(
        title='Daily Returns Distribution by Stock',
        xaxis_title='Daily Return',
        yaxis_title='Probability Density',
        template=PLOTLY_TEMPLATE,
        height=CHART_HEIGHT,
        width=CHART_WIDTH,
        barmode='overlay',
        legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99)
    )
    
    fig.show()
    fig.write_html('../visualizations/04_returns_distribution.html')
    print('Saved: visualizations/04_returns_distribution.html')


create_returns_distribution(returns_df)

In [ ]:
# Returns statistics summary
returns_stats = pd.DataFrame({
    'Mean Daily Return': returns_df.mean(),
    'Std Dev (Daily)': returns_df.std(),
    'Annualized Return': returns_df.mean() * 252,
    'Annualized Volatility': returns_df.std() * np.sqrt(252),
    'Sharpe Ratio (Rf=0)': (returns_df.mean() * 252) / (returns_df.std() * np.sqrt(252)),
    'Skewness': returns_df.skew(),
    'Kurtosis': returns_df.kurtosis(),
    'Min Daily Return': returns_df.min(),
    'Max Daily Return': returns_df.max(),
    'Max Drawdown': pd.Series({
        ticker: ((stock_data[ticker]['Adj Close'] / stock_data[ticker]['Adj Close'].cummax()) - 1).min()
        for ticker in active_tickers
    })
})

print('RETURNS STATISTICS SUMMARY')
print('=' * 80)
returns_stats.round(4)

## 8. Volatility and Correlation

### Rolling Volatility

Annualized rolling volatility (21-day window) shows how **risk evolves over time**. Key observations:

- Volatility tends to cluster: high-volatility periods persist
- Spikes often correspond to earnings announcements, macroeconomic events, or market crises
- The annualization factor `sqrt(252)` converts daily volatility to yearly terms (252 trading days/year)

In [ ]:
def create_volatility_chart(volatility_df):
    """
    Create rolling volatility comparison chart for all tickers.
    
    Parameters
    ----------
    volatility_df : pd.DataFrame
        DataFrame with rolling volatility for each ticker.
    """
    fig = go.Figure()
    
    for i, ticker in enumerate(volatility_df.columns):
        color = get_color(ticker, i)
        
        fig.add_trace(go.Scatter(
            x=volatility_df.index,
            y=volatility_df[ticker],
            name=ticker,
            line=dict(color=color, width=1.5),
            hovertemplate=(
                f'{ticker}<br>'
                'Date: %{x}<br>'
                'Volatility: %{y:.1%}'
                '<extra></extra>'
            )
        ))
    
    # Average volatility reference line
    avg_vol = volatility_df.mean().mean()
    fig.add_hline(
        y=avg_vol, line_dash='dot', line_color='gray',
        annotation_text=f'Avg: {avg_vol:.1%}', annotation_position='top right'
    )
    
    fig.update_layout(
        title='21-Day Rolling Annualized Volatility',
        xaxis_title='Date',
        yaxis_title='Annualized Volatility',
        yaxis_tickformat='.0%',
        template=PLOTLY_TEMPLATE,
        height=CHART_HEIGHT,
        width=CHART_WIDTH,
        hovermode='x unified',
        legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
    )
    
    fig.show()
    fig.write_html('../visualizations/05_rolling_volatility.html')
    print('Saved: visualizations/05_rolling_volatility.html')


create_volatility_chart(volatility_df)

### Correlation Heatmap

The correlation matrix of daily returns shows how closely stocks move together:

- **Correlation = 1.0**: Perfect positive correlation (move in lockstep)
- **Correlation = 0.0**: No relationship
- **Correlation = -1.0**: Perfect negative correlation (move in opposite directions)

For portfolio diversification, investors seek stocks with **low or negative correlation** to reduce overall portfolio risk.

In [ ]:
def create_correlation_heatmap(returns_df):
    """
    Create interactive correlation heatmap of daily returns.
    
    Parameters
    ----------
    returns_df : pd.DataFrame
        DataFrame with daily returns for each ticker.
    """
    corr_matrix = returns_df.corr()
    
    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns.tolist(),
        y=corr_matrix.index.tolist(),
        colorscale='RdBu_r',
        zmin=-1, zmax=1,
        zmid=0,
        text=corr_matrix.round(3).values,
        texttemplate='%{text}',
        textfont=dict(size=14),
        hovertemplate='%{x} vs %{y}<br>Correlation: %{z:.3f}<extra></extra>',
        colorbar=dict(title='Correlation')
    ))
    
    fig.update_layout(
        title='Stock Returns Correlation Matrix',
        template=PLOTLY_TEMPLATE,
        height=500,
        width=600,
        xaxis_title='',
        yaxis_title='',
        yaxis_autorange='reversed'
    )
    
    fig.show()
    fig.write_html('../visualizations/06_correlation_heatmap.html')
    print('Saved: visualizations/06_correlation_heatmap.html')
    
    # Print correlation rankings
    pairs = []
    cols = corr_matrix.columns.tolist()
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            pairs.append({
                'Pair': f'{cols[i]} - {cols[j]}',
                'Correlation': corr_matrix.iloc[i, j]
            })
    pairs_df = pd.DataFrame(pairs).sort_values('Correlation', ascending=False)
    
    print('\nCorrelation Rankings:')
    print(pairs_df.to_string(index=False))


create_correlation_heatmap(returns_df)

In [ ]:
# Rolling correlation between the most correlated pair
corr_matrix = returns_df.corr()

# Find the most correlated pair (excluding diagonal)
corr_values = corr_matrix.values.copy()
np.fill_diagonal(corr_values, 0)
max_idx = np.unravel_index(corr_values.argmax(), corr_values.shape)
ticker_a = corr_matrix.columns[max_idx[0]]
ticker_b = corr_matrix.columns[max_idx[1]]

rolling_corr = returns_df[ticker_a].rolling(window=60).corr(returns_df[ticker_b])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=rolling_corr.index, y=rolling_corr,
    name=f'{ticker_a} vs {ticker_b}',
    line=dict(color=get_color(ticker_a), width=1.5),
    fill='tozeroy',
    fillcolor='rgba(100,100,255,0.1)'
))

mean_corr = rolling_corr.mean()
fig.add_hline(
    y=mean_corr, line_dash='dash', line_color='red',
    annotation_text=f'Mean: {mean_corr:.3f}'
)

fig.update_layout(
    title=f'60-Day Rolling Correlation: {ticker_a} vs {ticker_b}',
    xaxis_title='Date',
    yaxis_title='Correlation',
    template=PLOTLY_TEMPLATE,
    height=400,
    width=CHART_WIDTH
)

fig.show()
print(f'Most correlated pair: {ticker_a} vs {ticker_b}')
print(f'Average 60-day rolling correlation: {mean_corr:.3f}')

## 9. Volume Analysis

Trading volume indicates the **intensity of market activity**:

- **High volume + price increase** = Strong buying conviction (bullish)
- **High volume + price decrease** = Strong selling pressure (bearish)
- **Low volume** = Lack of conviction; trend may be weak

The 20-day volume moving average provides a baseline for identifying **unusual trading activity**.

In [ ]:
def create_volume_analysis(stock_data, tickers):
    """
    Create volume analysis with daily bars and 20-day moving average per ticker.
    
    Parameters
    ----------
    stock_data : dict
        Dictionary of ticker -> DataFrame.
    tickers : list
        List of tickers to include.
    """
    fig = make_subplots(
        rows=len(tickers), cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=[f'{t} Daily Volume' for t in tickers]
    )
    
    for i, ticker in enumerate(tickers, 1):
        df = stock_data[ticker]
        color = get_color(ticker, i - 1)
        
        # Volume bars
        fig.add_trace(
            go.Bar(
                x=df.index,
                y=df['Volume'],
                name=f'{ticker} Volume',
                marker_color=color,
                opacity=0.5,
                showlegend=True
            ),
            row=i, col=1
        )
        
        # 20-day volume MA overlay
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df['Volume_SMA_20'],
                name=f'{ticker} 20-Day Avg',
                line=dict(color='black', width=1.5),
                showlegend=False
            ),
            row=i, col=1
        )
        
        fig.update_yaxes(title_text='Volume', row=i, col=1)
    
    fig.update_layout(
        title='Volume Analysis: Daily Volume with 20-Day Moving Average',
        template=PLOTLY_TEMPLATE,
        height=250 * len(tickers),
        width=CHART_WIDTH,
        showlegend=True,
        legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99)
    )
    
    fig.show()
    fig.write_html('../visualizations/07_volume_analysis.html')
    print('Saved: visualizations/07_volume_analysis.html')


create_volume_analysis(stock_data, active_tickers)

In [ ]:
# Volume statistics
print('VOLUME STATISTICS')
print('=' * 80)

volume_stats = pd.DataFrame({
    'Avg Daily Volume': {t: stock_data[t]['Volume'].mean() for t in active_tickers},
    'Max Daily Volume': {t: stock_data[t]['Volume'].max() for t in active_tickers},
    'Min Daily Volume': {t: stock_data[t]['Volume'].min() for t in active_tickers},
    'Volume Std Dev': {t: stock_data[t]['Volume'].std() for t in active_tickers},
    'Latest Volume': {t: stock_data[t]['Volume'].iloc[-1] for t in active_tickers},
    'Relative Volume': {
        t: stock_data[t]['Volume'].iloc[-1] / stock_data[t]['Volume'].mean()
        for t in active_tickers
    }
})

volume_stats.style.format({
    'Avg Daily Volume': '{:,.0f}',
    'Max Daily Volume': '{:,.0f}',
    'Min Daily Volume': '{:,.0f}',
    'Volume Std Dev': '{:,.0f}',
    'Latest Volume': '{:,.0f}',
    'Relative Volume': '{:.2f}x'
})

## 10. Statistics Dashboard

A comprehensive performance summary combining:
- **Total Return**: Overall price appreciation over the period
- **Risk vs Return**: Annualized return plotted against annualized volatility
- **Maximum Drawdown**: Largest peak-to-trough decline (worst-case scenario)
- **Sharpe Ratio**: Risk-adjusted return (higher = better return per unit of risk)

In [ ]:
def create_statistics_dashboard(stock_data, returns_df, tickers):
    """
    Create a 4-panel statistics dashboard.
    
    Parameters
    ----------
    stock_data : dict
        Dictionary of ticker -> DataFrame.
    returns_df : pd.DataFrame
        Combined daily returns DataFrame.
    tickers : list
        List of active tickers.
    """
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Total Return by Stock',
            'Risk vs Return (Annualized)',
            'Maximum Drawdown',
            'Sharpe Ratio Comparison'
        ),
        specs=[[{'type': 'bar'}, {'type': 'scatter'}],
               [{'type': 'bar'}, {'type': 'bar'}]]
    )
    
    colors_list = [get_color(t, i) for i, t in enumerate(tickers)]
    
    # Panel 1: Total return
    total_returns = {
        t: (stock_data[t]['Adj Close'].iloc[-1] / stock_data[t]['Adj Close'].iloc[0] - 1) * 100
        for t in tickers
    }
    
    fig.add_trace(go.Bar(
        x=list(total_returns.keys()),
        y=list(total_returns.values()),
        marker_color=colors_list,
        text=[f'{v:.1f}%' for v in total_returns.values()],
        textposition='outside',
        showlegend=False
    ), row=1, col=1)
    
    # Panel 2: Risk vs Return scatter
    ann_returns = returns_df.mean() * 252 * 100
    ann_volatility = returns_df.std() * np.sqrt(252) * 100
    
    for i, ticker in enumerate(tickers):
        fig.add_trace(go.Scatter(
            x=[ann_volatility[ticker]],
            y=[ann_returns[ticker]],
            mode='markers+text',
            text=[ticker],
            textposition='top center',
            marker=dict(size=15, color=get_color(ticker, i)),
            name=ticker,
            showlegend=False
        ), row=1, col=2)
    
    # Panel 3: Maximum drawdown
    max_drawdowns = {
        t: ((stock_data[t]['Adj Close'] / stock_data[t]['Adj Close'].cummax()) - 1).min() * 100
        for t in tickers
    }
    
    fig.add_trace(go.Bar(
        x=list(max_drawdowns.keys()),
        y=list(max_drawdowns.values()),
        marker_color=colors_list,
        text=[f'{v:.1f}%' for v in max_drawdowns.values()],
        textposition='outside',
        showlegend=False
    ), row=2, col=1)
    
    # Panel 4: Sharpe ratio
    sharpe_ratios = (returns_df.mean() * 252) / (returns_df.std() * np.sqrt(252))
    
    fig.add_trace(go.Bar(
        x=tickers,
        y=[sharpe_ratios[t] for t in tickers],
        marker_color=colors_list,
        text=[f'{sharpe_ratios[t]:.2f}' for t in tickers],
        textposition='outside',
        showlegend=False
    ), row=2, col=2)
    
    fig.update_layout(
        title='Stock Market Analysis: Performance Dashboard',
        template=PLOTLY_TEMPLATE,
        height=700,
        width=CHART_WIDTH,
    )
    
    fig.update_yaxes(title_text='Return (%)', row=1, col=1)
    fig.update_yaxes(title_text='Ann. Return (%)', row=1, col=2)
    fig.update_xaxes(title_text='Ann. Volatility (%)', row=1, col=2)
    fig.update_yaxes(title_text='Drawdown (%)', row=2, col=1)
    fig.update_yaxes(title_text='Sharpe Ratio', row=2, col=2)
    
    fig.show()
    fig.write_html('../visualizations/08_statistics_dashboard.html')
    print('Saved: visualizations/08_statistics_dashboard.html')


create_statistics_dashboard(stock_data, returns_df, active_tickers)

In [ ]:
# Comprehensive performance summary table
summary = pd.DataFrame(index=active_tickers)

summary['Latest Price'] = [stock_data[t]['Adj Close'].iloc[-1] for t in active_tickers]
summary['Period High'] = [stock_data[t]['Adj Close'].max() for t in active_tickers]
summary['Period Low'] = [stock_data[t]['Adj Close'].min() for t in active_tickers]
summary['Total Return (%)'] = [
    (stock_data[t]['Adj Close'].iloc[-1] / stock_data[t]['Adj Close'].iloc[0] - 1) * 100
    for t in active_tickers
]
summary['Ann. Return (%)'] = [returns_df[t].mean() * 252 * 100 for t in active_tickers]
summary['Ann. Volatility (%)'] = [returns_df[t].std() * np.sqrt(252) * 100 for t in active_tickers]
summary['Sharpe Ratio'] = summary['Ann. Return (%)'] / summary['Ann. Volatility (%)']
summary['Max Drawdown (%)'] = [
    ((stock_data[t]['Adj Close'] / stock_data[t]['Adj Close'].cummax()) - 1).min() * 100
    for t in active_tickers
]
summary['Avg Daily Volume'] = [stock_data[t]['Volume'].mean() for t in active_tickers]

print('COMPREHENSIVE PERFORMANCE SUMMARY')
print('=' * 100)
summary.round(2)

## 11. Key Findings

### Performance Summary

The analysis of 5 major tech stocks over a 2-year period reveals distinct risk-return profiles. Refer to the **Statistics Dashboard** above for the visual summary.

### Technical Signals

- Moving average crossovers (Golden Cross / Death Cross) were detected and annotated in the **Moving Averages** chart
- Stocks trading above their 200-day SMA are considered in a long-term uptrend

### Correlation Insights

- The **Correlation Heatmap** shows how closely these tech stocks move together
- High correlation among tech stocks suggests limited diversification within the sector
- The **Rolling Correlation** chart shows that correlations are not static and can vary significantly over time

### Volume Patterns

- Relative volume (current vs. 20-day average) helps identify stocks with unusual recent activity
- Volume spikes often precede or accompany significant price movements

---

## Skills Demonstrated

| Skill | Application |
|-------|------------|
| **API Integration** | Yahoo Finance data acquisition with per-ticker error handling and CSV caching |
| **Data Engineering** | Time-series preprocessing, feature engineering (returns, MAs, volatility) |
| **Interactive Visualization** | 8 publication-quality Plotly charts with tooltips, zoom, and range selection |
| **Financial Domain Knowledge** | Technical indicators, risk metrics (Sharpe, drawdown), correlation analysis |
| **Clean Code** | Modular functions with docstrings, consistent styling, one-line customization |

---

**Author:** [Your Name]  
**Date:** February 2026  
**Portfolio Project:** Day 2 of 30

In [ ]:
# Session summary
print('=' * 60)
print('ANALYSIS COMPLETE')
print('=' * 60)
print(f'\nStocks analyzed: {", ".join(active_tickers)}')
print(f'Trading days analyzed: {len(prices_df):,}')
print(f'Date range: {prices_df.index[0].strftime("%Y-%m-%d")} to {prices_df.index[-1].strftime("%Y-%m-%d")}')
print(f'Interactive visualizations created: 8')
print(f'\nOutput files saved to visualizations/:')

viz_files = [
    f'01_candlestick_{active_tickers[0]}.html',
    f'02_moving_averages_{active_tickers[0]}.html',
    '03_normalized_comparison.html',
    '04_returns_distribution.html',
    '05_rolling_volatility.html',
    '06_correlation_heatmap.html',
    '07_volume_analysis.html',
    '08_statistics_dashboard.html',
]

for f in viz_files:
    print(f'  - {f}')

print(f'\nAll charts are interactive - hover, zoom, and pan to explore!')
print(f'\nTo analyze different stocks, change TICKERS in the first code cell and re-run all.')